In [ ]:
from langchain_ollama.llms import OllamaLLM

def llm_model ():
    model_name = "llama3.1"
    
    llm = OllamaLLM(
        model=model_name
    )
    
    return llm
    
    
    
    

In [ ]:
llm = llm_model()


In [ ]:

llm.invoke("Hi waasup")

In [1]:
from langchain_ollama.llms import OllamaLLM
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore
from langchain_classic.retrievers import ParentDocumentRetriever

#define llm model
def llm_model ():
    
    try:
        model_name = "llama3.1"
        
        llm = OllamaLLM(
            model=model_name,
            temperature=0.5,
        )
        
        return llm
    
    except Exception as e:
        print(f"Unknown error: {e}")
        return None

#define embedding model
def embeddings():
    
    try:
        embed_model = HuggingFaceEmbeddings(
            model = "BAAI/bge-m3"
        )
        
        return embed_model
    
    except Exception as e:
        print(f"Unknown error: {e}")
        return None

#define PDF loader
def document_loader(file_path):
    
    try:
        if not file_path:
            raise ValueError("file path is empty or none")
        
        pfdLoader = PyPDFLoader(file_path=file_path)
        PdfData = pfdLoader.load()
        return PdfData
    
    except ValueError as e:
        print(f"value error: {e}")  
        return None
    except Exception as e:
        print(f"Unknown error: {e}")
        return None

#define text splitter
def text_splitters(data, chunk_size, chunk_overlap):
    
    try:
        
        if data is None:
            raise ValueError("file is none")
        
        if isinstance(data, list) and len(data) ==0:
            raise ValueError ("file is empty")
        
        splitter = RecursiveCharacterTextSplitter(
            separators=["\n\n", "\n", " ", ""],
            chunk_size = chunk_size,
            chunk_overlap=chunk_overlap,
            length_function = len
        )
        
        chunks = splitter.split_documents(data)
        return chunks
    
    except ValueError as e:
        print(f"value error:{e}")
        return None
    
    except Exception as e:
        print(f"Unknown error: {e}")
        return None
    

#define vector database
def vector_database(documents, embedding_model):
    
    try:
        if documents is None:
            raise ValueError("document is empty or None")
        
        chroma_db = Chroma.from_documents(documents=documents,
                                          embedding=embedding_model,
                                          persist_directory="chroma_db")
        
        return chroma_db
        
    except ValueError as e:
        print(f"value error: {e}")
        return None
    
    except Exception as e:
        print(f"Unknown error: {e}")
        return None
    
def retriever_call(file_path,parent_chunk_size=2000, 
            parent_chunk_overlap=400, child_chunk_size=500, child_chunk_overlap=50):
    
    try:
        
        if not file_path:
            raise ValueError ("Document is empty or none")
        
        documents = document_loader(file_path)
        
        parent_splitter = RecursiveCharacterTextSplitter(chunk_size = parent_chunk_size,
                                                chunk_overlap=parent_chunk_overlap,
                                                length_function=len)
        
        child_splitter = RecursiveCharacterTextSplitter(chunk_size = child_chunk_size,
                                                chunk_overlap=child_chunk_overlap,
                                                length_function=len)
        
        embedding_model = embeddings()
        
        vectore_store = Chroma(
            collection_name="split_parent", embedding_function=embedding_model
        )
        
        doc_store = InMemoryStore()
        
        retriever = ParentDocumentRetriever(
            vectorstore=vectore_store,
            docstore=doc_store,
            parent_splitter=parent_splitter,
            child_splitter=child_splitter
        )
        
        retriever.add_documents(documents=documents)
        return retriever
    
    except ValueError as e:
        print(f"value error: {e}")
        return None
    
    except Exception as e:
        print(f"unknown error: {e}")
        return None
    

d:\Projects\docuMind-aI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Vish\AppData\Local\Temp\ipykernel_14656\4042682035.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
import gradio as gr


def retriever_qa(file_path, query):

    try:
        
        if isinstance(file_path,dict):
            file_path = file_path.get("path") or file_path.get("name")
        
        if not file_path:
            raise ValueError("file path in invalid or none")
        
        llm = llm_model()
        
        retriever_obj = retriever_call(file_path)

        prompt = ChatPromptTemplate.from_template("""
            
            Answer using only the context below.

            Context:
            {context}

            Question:
            {input}
            """)
        
        doc_chain = create_stuff_documents_chain(llm=llm,prompt=prompt)
        
        #connect retriever
        qa_chain = create_retrieval_chain(retriever=retriever_obj,
                                          combine_docs_chain=doc_chain)
        
        response = qa_chain.invoke({"input":query})
        return response.get("answer") or response.get("result") or str(response)
    
    except ValueError as e:
        print(f"value error: {e}")
        return None

    except Exception as e:
        print(f"unknown error:{e}")
        return None
    
def gradio_interface():

    interface = gr.Interface(
        fn=retriever_qa,
        inputs=[
            gr.File(label="Upload PDF File", 
                    file_count="single", file_types=['.pdf'],type="filepath"),
            gr.TextArea(label="Input Query",placeholder="What's in your mind...")
        ],
        outputs= gr.TextArea(label="Response", placeholder="Waiting for response..."),
        title="DocuMind AI",
        description="Upload a PDF document and ask any question. The chatbot will try to answer using the provided document."
    )
    
    interface.launch(share=True)
    

In [3]:
file = "../files/EC 8206_ lecture_02.pdf"
retriever_qa(file, "what is this doxument about?")

d:\Projects\docuMind-aI\venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


'This document appears to be a collection of examples and exercises related to Functional Programming (FP) in Haskell. It provides step-by-step solutions to various problems, including solving quadratic equations, calculating compound interest, and finding sums and squares of numbers divisible by 3 or 5. The document seems to be a tutorial or educational resource for learning Haskell programming.'